# t2i 评测集 V1 · 完整分析

第一版 t2i 评测集题库 = **200 题**（r10 验证批 10 题 + r11 大批量 190 题），
四图源产出（`gemini-3.1-flash-image` / `bagel-7B-MoT` / `Z-Image-Turbo` / `Qwen-Image-2512`），统一判分（`gpt-5.6-sol` × V2，判官自推隐式蕴含）。

| 项 | 值 |
|---|---|
| 题库 | `data/bench_v1/questions.jsonl`（qid 60001-60202，缺 60029/60036 为不可行概念替换） |
| 难度梯子 | L1:L2:L3 = 1:6:3（20/120/60），条目制 v6.0 协议（重标定后） |
| 抽样 | 296k 实例池中 29 域等概率均匀抽样（质量门 quality≥9 + identity + 短边≥768），每二级分支限 1 题，排除全部历史已用实例 |
| 出题 | `gpt-5.6-sol` × v6.0 协议，双向削峰（--peak-shave --lanes 5，主域豁免补丁） |
| 生图 | gemini（网关 4001）+ bagel（BAGEL-7B-MoT，512px/50步）+ Z-Image-Turbo（diffusers）+ Qwen-Image-2512（diffusers） |
| 判分 | V2：对齐 10 项 / 质量 8 项 / 美感 4 项，刻度 {0,1,2,NA}，支柱均值 0-100 |

分析维度：总览 → 难度剖面（均值 + 尾部双口径） → 域剖面 → 六统计字段剖面（组合/场景/跳/弱项/前提/知识域）→ facet 剖面 → 四模型 head-to-head → 逐题明细。
修改参数后 **务必硬重载**（Shutdown Kernel + 关标签重开）。


In [ ]:
import json, base64, html as H, io
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML

plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'Noto Sans CJK JP', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# cwd 自适应：仓库根 或 benchmark/t2i 均可
_B = Path('data/bench_v1')
if not _B.exists():
    _B = Path('benchmark/t2i/data/bench_v1')
assert _B.exists(), f'找不到 bench_v1：{_B.resolve()}（请在仓库根或 benchmark/t2i 下运行）'
B = _B
GEM, BAG = 'gemini-3.1-flash-image', 'bagel'
ZIM, QWEN = 'Z-Image-Turbo', 'Qwen-Image-2512'
PHI = {0: 0, 1: 60, 2: 100}
MODEL_COLOR = {GEM: '#2a7ab0', BAG: '#b06a20', ZIM: '#3a8a3a', QWEN: '#8a3a8a'}
LV_ORDER = ['L1', 'L2', 'L3']
ALL_MODELS = [GEM, BAG, ZIM, QWEN]

def jl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

qs = pd.DataFrame(jl(B / 'questions.jsonl'))

def load_scores(name):
    rows = jl(B / 'scores' / name)
    ok = pd.DataFrame([r for r in rows if not r.get('fail')])
    ok = ok.drop(columns=['level'], errors='ignore')  # 与题目表 level 冲突，题目表为权威
    return ok, sum(1 for r in rows if r.get('fail'))

sg, sg_fail = load_scores(f'scores_v60_V2_gpt-5.6-sol_{GEM}.jsonl')
sb, sb_fail = load_scores('scores_v60_V2_gpt-5.6-sol_bagel.jsonl')
sz, sz_fail = load_scores('scores_v60_V2_gpt-5.6-sol_Z-Image-Turbo.jsonl')
sq, sq_fail = load_scores('scores_v60_V2_gpt-5.6-sol_Qwen-Image-2512.jsonl')

# 域/分支/实例：qid 即 sample_id，回查三份抽样清单
samp = {}
for f in ('samples_v60_uniform.jsonl', 'samples_v60_uniform_r11.jsonl', 'samples_v60_uniform_r11b.jsonl'):
    for r in jl(B.parent / f):
        samp[int(r['sample_id'])] = r
qs['domain'] = qs.qid.astype(int).map(lambda i: samp[i]['l1'])
qs['branch'] = qs.qid.astype(int).map(lambda i: samp[i]['l2'])
qs['instance'] = qs.qid.astype(int).map(lambda i: samp[i]['instance'])

def with_level(df):   # 判分行并入层级/批次（全 200 题都应有分）
    return df.merge(qs[['qid', 'level', 'batch']], on='qid')

print(f'题库 {len(qs)} 题（qids {qs.qid.min()}..{qs.qid.max()}，唯一 {qs.qid.nunique()}）')
print(f'难度: {qs.level.value_counts().to_dict()}  | 批次: {qs.batch.value_counts().to_dict()}')
print(f'域覆盖: {qs.domain.nunique()}/29（池内 27 域；政治法律/宗教信仰两域质量门内无合格实例）')
print(f'二级分支: {qs.groupby(["domain","branch"]).ngroups} 个，无重复（每分支限 1 题）')
print(f'判分覆盖: {GEM} {len(sg)}/{len(qs)}（fail {sg_fail}），bagel {len(sb)}/{len(qs)}（fail {sb_fail}），{ZIM} {len(sz)}/{len(qs)}（fail {sz_fail}），{QWEN} {len(sq)}/{len(qs)}（fail {sq_fail}）')


## 1 · 总览：双图源三轴得分

三支柱均值与分布（0-100）。对齐=题面要求达成度（判官自推隐式蕴含），质量=物理/质感/瑕疵，美感=构图/色彩/光影。

In [ ]:
def pillar_table(df, name):
    return pd.DataFrame({name: {
        '对齐': round(df.alignment_score.mean(), 2),
        '质量': round(df.quality_score.mean(), 2),
        '美感': round(df.aesthetic_score.mean(), 2),
        'n': len(df)}})

ov = pd.concat([pillar_table(sg, GEM), pillar_table(sb, BAG), pillar_table(sz, ZIM), pillar_table(sq, QWEN)], axis=1)
display(ov)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, col, title in zip(axes, ('alignment_score', 'quality_score', 'aesthetic_score'),
                          ('对齐', '质量', '美感')):
    for m, df in ((GEM, sg), (BAG, sb), (ZIM, sz), (QWEN, sq)):
        ax.hist(df[col], bins=20, alpha=.55, label=m, color=MODEL_COLOR[m])
    ax.set_title(f'{title} 分布'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# r10/r11 批间稳定性（同协议同判分）
for m, df in ((GEM, sg), (BAG, sb), (ZIM, sz), (QWEN, sq)):
    t = with_level(df).groupby('batch')[
        ['alignment_score', 'quality_score', 'aesthetic_score']].mean().round(2)
    print(f'\n[{m}] 批间对比（r10=验证批10题 / r11=大批量190题）'); display(t)

## 2 · 难度剖面（均值 + 尾部双口径）

梯子有效性的检验要看两个口径：**均值单调**（越难分越低）与**失败尾单调**（<40 分占比越难越高）。
实测提示：SOTA 级模型（gemini）均值近乎平坦但失败尾严格单调——难度效应藏在尾部；
7B 模型（bagel）两个口径都单调。

In [ ]:
rows = []
for m, df in ((GEM, sg), (BAG, sb), (ZIM, sz), (QWEN, sq)):
    t = with_level(df)
    for lv in LV_ORDER:
        s = t[t.level == lv].alignment_score
        rows.append({'模型': m, '层级': lv, 'n': len(s),
                     'mean': s.mean(), 'median': s.median(), 'std': s.std(),
                     'min': s.min(), 'max': s.max(),
                     '<40分': int((s < 40).sum()), '<40率': (s < 40).mean(),
                     '≥70分': int((s >= 70).sum()), '≥70率': (s >= 70).mean()})
lv = pd.DataFrame(rows)
disp = lv.copy()
for c in ('mean', 'median', 'std', 'min', 'max'):
    disp[c] = disp[c].round(1)
for c in ('<40率', '≥70率'):
    disp[c] = (disp[c] * 100).round(0).astype(int).astype(str) + '%'
display(disp)

piv_mean = lv.pivot(index='层级', columns='模型', values='mean').reindex(LV_ORDER)
piv_fail = lv.pivot(index='层级', columns='模型', values='<40率').reindex(LV_ORDER)
for m in piv_mean.columns:
    vm, vf = piv_mean[m].dropna(), piv_fail[m].dropna()
    mono_m = all(vm.iloc[i] >= vm.iloc[i+1] for i in range(len(vm)-1))
    mono_f = all(vf.iloc[i] <= vf.iloc[i+1] for i in range(len(vf)-1))
    print(f'{m}: 均值 L1→L3 {vm.iloc[0]:.1f}→{vm.iloc[-1]:.1f}（落差 {vm.iloc[0]-vm.iloc[-1]:.1f}，单调 {"✓" if mono_m else "✗"}）'
          f' | 失败尾 <40: ' + '→'.join(f'{x:.0%}' for x in vf) +
          f'（单调 {"✓" if mono_f else "✗"}）')

fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.4))
x = np.arange(3)
ax = axes[0]
w = .36
for i, m in enumerate(piv_mean.columns):
    ax.bar(x + (i - .5) * w, piv_mean[m].values, w, label=m, color=MODEL_COLOR[m])
ax.set_xticks(x, [f'{l}（n={int(lv[lv["层级"]==l].n.max())}）' for l in LV_ORDER])
ax.set_ylabel('对齐分'); ax.set_title('均值剖面'); ax.legend(fontsize=8)
ax = axes[1]
for i, m in enumerate(piv_fail.columns):
    ax.bar(x + (i - .5) * w, piv_fail[m].values * 100, w, label=m, color=MODEL_COLOR[m])
ax.set_xticks(x, LV_ORDER); ax.set_ylabel('<40 分占比 %'); ax.set_title('失败尾剖面（越难应越高）'); ax.legend(fontsize=8)
ax = axes[2]
data, labels, colors = [], [], []
for m, df in ((GEM, sg), (BAG, sb), (ZIM, sz), (QWEN, sq)):
    t = with_level(df)
    for l in LV_ORDER:
        data.append(t[t.level == l].alignment_score.values)
        labels.append(f'{m[:3]}·{l}'); colors.append(MODEL_COLOR[m])
bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, showmeans=True,
                medianprops=dict(color='k', lw=.8))
for p, c in zip(bp['boxes'], colors):
    p.set_facecolor(c); p.set_alpha(.55)
ax.tick_params(axis='x', labelrotation=45, labelsize=7)
ax.axhline(40, color='#a33333', ls=':', lw=.8)
ax.set_title('对齐分箱线图（虚线=40 分失败线）'); ax.set_ylabel('对齐分')
plt.tight_layout(); plt.show()

# 逐层原始分（小样本层直观看全量）
for m, df in ((GEM, sg), (BAG, sb), (ZIM, sz), (QWEN, sq)):
    t = with_level(df)
    s = t[t.level == 'L1'].sort_values('alignment_score')
    print(f'[{m}] L1 全 {len(s)} 题（升序）: ' + ', '.join(f'{int(r.qid)}:{r.alignment_score:.0f}' for _, r in s.iterrows()))

## 3 · 域剖面

27 个知识域的对齐得分（均匀抽样 → 各域 n 6-13，含池内小域稀疏提示）。

In [ ]:
rows = []
for m, df in ((GEM, sg), (BAG, sb), (ZIM, sz), (QWEN, sq)):
    t = df.merge(qs[['qid', 'domain']], on='qid').groupby('domain')['alignment_score'].agg(['mean', 'count'])
    for d, r in t.iterrows():
        rows.append({'domain': d, 'model': m, 'mean': r['mean'], 'n': int(r['count'])})
dd = pd.DataFrame(rows)
piv = dd.pivot(index='domain', columns='model', values='mean')
ncnt = dd.groupby('domain')['n'].max()
piv = piv.loc[ncnt.sort_values().index]

fig, ax = plt.subplots(figsize=(10, max(6, .28 * len(piv))))
y = np.arange(len(piv))
w = .18
for i, m in enumerate(ALL_MODELS):
    if m in piv.columns:
        ax.barh(y + (i - 1.5) * w, piv[m].fillna(np.nan), w, label=m, color=MODEL_COLOR[m])
ax.set_yticks(y, [f'{d} ({n})' for d, n in ncnt[piv.index].items()], fontsize=8)
ax.set_xlabel('对齐分'); ax.set_title('各知识域对齐分（括号=题数）'); ax.legend()
ax.axvline(sg.alignment_score.mean(), ls=':', c=MODEL_COLOR[GEM], lw=1)
ax.axvline(sb.alignment_score.mean(), ls=':', c=MODEL_COLOR[BAG], lw=1)
plt.tight_layout(); plt.show()

best = piv.mean(axis=1).sort_values()
print('最難域（双模型均分最低）:'); display(best.head(5).round(1))
print('\n最易域:'); display(best.tail(5).round(1))

## 4 · 六统计字段剖面

出题时机的六个设计维度：组合主类 / 场景复杂度来源 / 跳类型 / 弱项 / 前提类别 / 知识域注入。
上排=题库分布（削峰验证），下排=各类对齐分（模型能力短板定位）。

In [ ]:
def field_profile(field, topn=14):
    cnt = Counter()
    for v in qs[field]:
        cnt.update(v if isinstance(v, list) else [v])
    cats = [c for c, _ in cnt.most_common(topn)]
    per = {m: {} for m in (GEM, BAG)}
    for m, df in ((GEM, sg), (BAG, sb)):
        t = df.merge(qs[['qid', field]], on='qid')
        for c in cats:
            vals = [r.alignment_score for _, r in t.iterrows()
                    if c in (r[field] if isinstance(r[field], list) else [r[field]])]
            per[m][c] = (round(np.mean(vals), 1), len(vals)) if vals else (np.nan, 0)
    return cnt, cats, per

fig, axes = plt.subplots(6, 2, figsize=(13, 26))
for ax_row, field in zip(axes, ('combo_type', 'scene_types', 'hop_types',
                                'weak_points', 'premise_types', 'knowledge_domains')):
    cnt, cats, per = field_profile(field)
    ax = ax_row[0]
    top = cnt.most_common(14)
    ax.barh([c for c, _ in top][::-1], [n for _, n in top][::-1], color='#557')
    for i, (c, n) in enumerate(top[::-1]):
        ax.text(n, i, f' {n}', va='center', fontsize=8)
    ax.set_title(f'{field} · 题库分布（题数）', fontsize=11)
    ax.tick_params(labelsize=8)
    ax = ax_row[1]
    y = np.arange(len(cats))
    w = .18
    for i, m in enumerate(ALL_MODELS):
        vals = [per[m][c][0] for c in cats]
        ax.barh(y + (i - 1.5) * w, vals, w, label=m, color=MODEL_COLOR[m])
    ax.set_yticks(y, cats, fontsize=8)
    ax.set_xlim(0, 100); ax.set_xlabel('对齐分')
    ax.set_title(f'{field} · 各类对齐分（括号=题数）', fontsize=11)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

mirror = sum(1 for c in qs.combo_type if c in ('光学媒介', '光学传播'))
print(f'削峰验证：镜面类组合 {mirror}/{len(qs)} = {mirror/len(qs):.0%}（批门槛 ≤20%，L3 豁免后受控）')
print(f'组合主类多样性：{qs.combo_type.nunique()} 类 | 弱项种类：{qs.weak_points.explode().nunique()} | 跳类型：{qs.hop_types.explode().nunique()}')

## 5 · facet 剖面

判分 22 子项（对齐 10 / 质量 8 / 美感 4），刻度 {0,1,2,N/A} 经 φ 映射（0/60/100）后取均值。子项短板=模型能力显微镜。

In [ ]:
def facet_frame(df, key):
    recs = []
    for _, r in df.iterrows():
        d = r.get(key) or {}
        recs.append({k: (v if isinstance(v, (int, float)) else np.nan) for k, v in d.items()})
    f = pd.DataFrame(recs)
    return f.apply(lambda s: s.map(PHI)), f.notna().mean(axis=0)  # φ 均值, 覆盖率

fig, axes = plt.subplots(1, 3, figsize=(14, 7))
for ax, (key, title) in zip(axes, (('alignment_scores', '对齐（10 项）'),
                                   ('quality_scores', '质量（8 项）'),
                                   ('aesthetic_scores', '美感（4 项）'))):
    fm, cov = {}, {}
    for m, df in ((GEM, sg), (BAG, sb), (ZIM, sz), (QWEN, sq)):
        fm[m], cov[m] = facet_frame(df, key)
    cols = list(fm[GEM].columns)
    y = np.arange(len(cols))
    w = .18
    for i, m in enumerate(ALL_MODELS):
        if m in fm:
            ax.barh(y + (i - 1.5) * w, [fm[m][c].mean() for c in cols], w, label=m, color=MODEL_COLOR[m])
    ax.set_yticks(y, [f'{c}\\nN/A {1-cov[GEM].get(c,0):.0%}' for c in cols], fontsize=7.5)
    ax.set_xlim(0, 100); ax.set_title(title, fontsize=11); ax.legend(fontsize=8)
    ax.axvline(60, ls=':', c='gray', lw=.8)
plt.tight_layout(); plt.show()

print('facet 短板定位（对齐项 φ 均值，双模型最低各 5 项）:')
for m, df in ((GEM, sg), (BAG, sb), (ZIM, sz), (QWEN, sq)):
    fm, _ = facet_frame(df, 'alignment_scores')
    print(f'  [{m}]', fm.mean().sort_values().head(5).round(1).to_dict())

## 6 · head-to-head：四模型两两对比

同题对齐分差（行模型 − 列模型）：胜负平矩阵 + gemini vs 其他三模型散点。


In [ ]:
# Pairwise head-to-head: wins/ties/losses across all 4 models
models_data = [(GEM, sg), (BAG, sb), (ZIM, sz), (QWEN, sq)]
n = len(models_data)
win_mtx = [[0]*n for _ in range(n)]
for i, (m1, d1) in enumerate(models_data):
    for j, (m2, d2) in enumerate(models_data):
        if i == j:
            win_mtx[i][j] = '-'
            continue
        h2 = d1[['qid', 'alignment_score']].merge(d2[['qid', 'alignment_score']], on='qid', suffixes=('_1', '_2'))
        h2['diff'] = h2.alignment_score_1 - h2.alignment_score_2
        w = (h2['diff'] > 1).sum()
        l = (h2['diff'] < -1).sum()
        t = (h2['diff'].abs() <= 1).sum()
        win_mtx[i][j] = f'{w}胜/{l}负/{t}平'

print('Head-to-head 矩阵（行=模型A，列=模型B）:')
header = f'{"":>20}' + ''.join(f'{m:>20}' for m, _ in models_data)
print(header)
for i, (m, _) in enumerate(models_data):
    row = f'{m:>20}' + ''.join(f'{win_mtx[i][j]:>20}' for j in range(n))
    print(row)

# Scatter plot matrix: alignment scores
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
pairs = [((GEM, sg), (BAG, sb)), ((GEM, sg), (ZIM, sz)), ((GEM, sg), (QWEN, sq))]
for ax, (m1, d1, m2, d2) in zip(axes, [(p[0][0], p[0][1], p[1][0], p[1][1]) for p in pairs]):
    h2 = d1[['qid', 'alignment_score']].merge(d2[['qid', 'alignment_score']], on='qid', suffixes=('_1', '_2'))
    ax.scatter(h2.alignment_score_2, h2.alignment_score_1, s=9, alpha=.55, c=MODEL_COLOR[m1])
    ax.plot([0, 100], [0, 100], 'r:', lw=1)
    ax.set_xlabel(m2); ax.set_ylabel(m1)
    w = (h2.alignment_score_1 - h2.alignment_score_2 > 1).sum()
    l = (h2.alignment_score_1 - h2.alignment_score_2 < -1).sum()
    ax.set_title(f'{m1} vs {m2}: {w}胜/{l}负')
plt.tight_layout(); plt.show()


## 7 · 逐题明细

按四模型最低对齐分升序（最难的头 K 题）：题面 / 参照图 / 四模型产出图 / 三轴分 + 子项判语（判语与打分同行展示）。
图片为缩略图内嵌（笔记本体积可控）；改 `K` / `ORDER` 后重跑本 cell。


In [ ]:
K = 20          # 展示题数
ORDER = 'worst' # 'worst' | 'best' | 'random'
THUMB = 320     # 缩略图最长边 px
GC = {0: '#a33333', 1: '#b07020', 2: '#2a7a4b'}

def b64(p, maxw=THUMB):
    p = Path(p)
    if not p.exists():
        return '<div style="color:#999;font-size:11px">缺图</div>'
    try:
        im = Image.open(p); im.thumbnail((maxw, maxw))
        buf = io.BytesIO(); im.convert('RGB').save(buf, 'JPEG', quality=82)
        d = base64.b64encode(buf.getvalue()).decode()
        return f'<img src="data:image/jpeg;base64,{d}" style="border:1px solid #ccc"/>'
    except Exception as e:
        return f'<div style="color:#999;font-size:11px">图读取失败 {H.escape(str(e)[:40])}</div>'

SUBS = ['alignment_score', 'quality_score', 'aesthetic_score',
       'alignment_scores', 'quality_scores', 'aesthetic_scores',
       'alignment_reasons', 'quality_reasons', 'aesthetic_reasons']
MODEL_SUFFIX = {GEM: '_g', BAG: '_b', ZIM: '_z', QWEN: '_q'}
# Build merged dataframe with all 4 models
merged = qs[['qid', 'level', 'instance', 'domain', 'branch', 'combo_type', 'weak_points', 'gen_prompt']].copy()
for m, df in ((GEM, sg), (BAG, sb), (ZIM, sz), (QWEN, sq)):
    suffix = MODEL_SUFFIX[m]
    merged = merged.merge(df[['qid'] + SUBS], on='qid', how='inner', suffixes=('', suffix))
    # Rename suffixed columns
    for s in SUBS:
        if s + suffix in merged.columns:
            merged = merged.rename(columns={s + suffix: s + suffix})

# Use min alignment across all 4 models for ordering
align_cols = [f'alignment_score{MODEL_SUFFIX[m]}' for m in ALL_MODELS if f'alignment_score{MODEL_SUFFIX[m]}' in merged.columns]
# Fall back to available columns
align_cols = [c for c in align_cols if c in merged.columns]
merged['minalign'] = merged[align_cols].min(axis=1) if align_cols else 0

sel = merged.nsmallest(K, 'minalign') if ORDER == 'worst' else (
      merged.nlargest(K, 'minalign') if ORDER == 'best' else merged.sample(K, random_state=1))

parts = []
for _, q in sel.iterrows():
    sid = int(q.qid)
    ref = B / 'samples' / Path(samp[sid]['image']).name
    img_paths = {
        GEM: B / 'gemini' / 'imgs' / GEM / f'{q.qid}.png',
        BAG: B / 'bagel' / 'imgs' / f'{q.qid}.png',
        ZIM: B / 'zimage' / 'imgs' / f'{q.qid}.png',
        QWEN: B / 'qwen2512' / 'imgs' / f'{q.qid}.png',
    }
    scores_str = ' · '.join(f'{m} {q[f"alignment_score{MODEL_SUFFIX[m]}"]:.0f}/{q[f"quality_score{MODEL_SUFFIX[m]}"]:.0f}/{q[f"aesthetic_score{MODEL_SUFFIX[m]}"]:.0f}'
                            for m in ALL_MODELS if f'alignment_score{MODEL_SUFFIX[m]}' in q.index)
    head = (f"<div style='margin:14px 0 4px'><b>{q.qid}</b> · {q.instance} · {q.domain} / {q.branch}"
            f" · <b>{q.level}</b> · 组合={q.combo_type} · 弱点={'、'.join(q.weak_points[:3])}"
            f" · {scores_str}</div>")
    gp_html = (f"<div style='white-space:pre-wrap;font-size:12px;margin:2px 0 6px;"
                f"background:#f6f6f8;border-left:3px solid #99a;padding:6px 10px'>"
                f"{H.escape(str(q.gen_prompt))}</div>")
    def _sc(v):
        c = '#2a7a4b' if v >= 60 else ('#b07020' if v >= 40 else '#a33333')
        return f'<td style="color:{c};font-weight:bold;text-align:center">{v:.1f}</td>'
    # Score table header
    score_tbl = '<table style="font-size:12px;border-collapse:collapse;margin:4px 0"><tr><td></td>' + ''.join(f'<th style="padding:1px 14px">{x}</th>' for x in ('对齐', '质量', '美感')) + '</tr>'
    for m in ALL_MODELS:
        s = MODEL_SUFFIX[m]
        if f'alignment_score{s}' in q.index:
            score_tbl += (f'<tr><td style="color:#888">{m}</td>' +
                          _sc(q[f'alignment_score{s}']) + _sc(q[f'quality_score{s}']) + _sc(q[f'aesthetic_score{s}']) + '</tr>')
    score_tbl += '</table>'
    # Images row
    imgs = '<table><tr><td>参照<br/>' + b64(ref) + '</td>'
    for m in ALL_MODELS:
        imgs += f'<td>{m}<br/>{b64(img_paths[m])}</td>'
    imgs += '</tr></table>'
    # Detail rows
    rows_html = ['<table style="font-size:11px;border-collapse:collapse">']
    for m in ALL_MODELS:
        s = MODEL_SUFFIX[m]
        for key, name in (('alignment', '对齐'), ('quality', '质量'), ('aesthetic', '美感')):
            sc_d = q.get(f'{key}_scores{s}') or {}
            rs_d = q.get(f'{key}_reasons{s}') or {}
            for f, v in sc_d.items():
                c = GC.get(v, '#999')
                vs = 'N/A' if not isinstance(v, (int, float)) else f'{v}'
                rows_html.append(
                    f"<tr><td style='color:#888;white-space:nowrap'>{m[:6]}·{name}</td>"
                    f"<td style='color:#888;white-space:nowrap'>{f}</td>"
                    f"<td style='color:{c};font-weight:bold;text-align:center'>{vs}</td>"
                    f"<td>{H.escape(str(rs_d.get(f, '')))[:260]}</td></tr>")
    rows_html.append('</table>')
    parts.append(head + gp_html + score_tbl + imgs + ''.join(rows_html))
display(HTML('<div style="max-height:1200px;overflow-y:auto">' + ''.join(parts) + '</div>'))


## 8 · 结论摘要

自动汇总（数值随数据自动更新）。


In [ ]:
# Compute stats for all 4 models
models_data = [(GEM, sg), (BAG, sb), (ZIM, sz), (QWEN, sq)]
stats = []
for m, df in models_data:
    t = with_level(df)
    a = df.alignment_score.mean()
    q = df.quality_score.mean()
    ae = df.aesthetic_score.mean()
    lv = t.groupby('level')['alignment_score'].mean()
    fail_rate = t.assign(f=lambda d: d.alignment_score < 40).groupby('level')['f'].mean()
    stats.append({'model': m, 'align': a, 'quality': q, 'aesthetic': ae,
                  'L1': lv.get('L1', 0), 'L2': lv.get('L2', 0), 'L3': lv.get('L3', 0),
                  'fail_L1': fail_rate.get('L1', 0), 'fail_L2': fail_rate.get('L2', 0), 'fail_L3': fail_rate.get('L3', 0)})

lines = []
lines.append('=== t2i 评测集 V1 · 200 题四模型结论 ===')
lines.append('1. 总体对齐分: ' + ' / '.join(f"{s['model']} {s['align']:.1f}" for s in stats))
lines.append('   总体质量分: ' + ' / '.join(f"{s['model']} {s['quality']:.1f}" for s in stats))
lines.append('   总体美感分: ' + ' / '.join(f"{s['model']} {s['aesthetic']:.1f}" for s in stats))
lines.append('2. 难度梯子（均值口径 L1→L3）: ' + ' / '.join(f"{s['model']} {s['L1']:.1f}→{s['L3']:.1f}" for s in stats))
lines.append('3. 难度梯子（失败尾 <40 分）: ' + ' / '.join(f"{s['model']} {s['fail_L1']:.0%}→{s['fail_L3']:.0%}" for s in stats))
lines.append(f'4. 题库分布：镜面 {mirror/len(qs):.0%}（≤20% 门槛内）、组合 {qs.combo_type.nunique()} 类、弱项 {qs.weak_points.explode().nunique()} 种')
lines.append('5. 判分异常: ' + ' / '.join(f"{s['model']} fail {eval(s['model'].split('-')[0].lower() + '_fail')}" for s in stats) + ' 行')
print('\n'.join(lines))
